<a href="https://colab.research.google.com/github/jesildabraganca0511/NLP/blob/main/mini-projects/week1_sms_spam_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub


In [2]:
path=kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Path to dataset files: /kaggle/input/sms-spam-collection-dataset


In [3]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv('/kaggle/input/sms-spam-collection-dataset/spam.csv', encoding='latin-1')


In [8]:
df[:4]

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN


In [9]:
df=df[["v1","v2"]]

In [17]:
df.columns = ['label', 'message']


In [18]:
df

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


check for null values

In [11]:
df.isnull().sum()

,0
v1,0
v2,0


check for duplicates

In [14]:
df.duplicated().sum()

np.int64(403)

clean dupliactes

In [15]:
df.drop_duplicates(keep='first',inplace=True)

check imbalance

In [30]:
df["label"].value_counts()

,count
label,
ham,4516
spam,653


In [31]:
df['message'] = df["message"].str.lower()

In [21]:
import re

In [32]:
df['message'] = df['message'].apply(lambda x: re.sub(r'[^\w\s]', '', x))


In [33]:
import spacy

In [34]:
nlp=spacy.load("en_core_web_sm")

In [42]:
def tokenize_message(raw_text):
  doc = nlp(raw_text)
  tokens = [token.lemma_ for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha]
  return tokens

In [43]:
df["message_tokens"]=df["message"].apply(tokenize_message)

In [44]:
df.head()

,label,message,message_tokens
0,ham,go until jurong point crazy available only in ...,"[jurong, point, crazy, available, bugis, n, gr..."
1,ham,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, wkly, comp, win, fa, cup, final,..."
3,ham,u dun say so early hor u c already then say,"[u, dun, early, hor, u, c]"
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, not, think, go, usf, live]"


stringing it together

In [45]:
df["processed_message"]= df["message_tokens"].apply(lambda x: " ".join(x))

In [47]:
df.head()

,label,message,message_tokens,processed_message
0,ham,go until jurong point crazy available only in ...,"[jurong, point, crazy, available, bugis, n, gr...",jurong point crazy available bugis n great wor...
1,ham,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]",ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, wkly, comp, win, fa, cup, final,...",free entry wkly comp win fa cup final tkts tex...
3,ham,u dun say so early hor u c already then say,"[u, dun, early, hor, u, c]",u dun early hor u c
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, not, think, go, usf, live]",nah not think go usf live


In [48]:
df_final = df[['label', 'processed_message']].copy()
display(df_final.head())

,label,processed_message
0,ham,jurong point crazy available bugis n great wor...
1,ham,ok lar joking wif u oni
2,spam,free entry wkly comp win fa cup final tkts tex...
3,ham,u dun early hor u c
4,ham,nah not think go usf live


Feature Extraction